# Segmentation

Per-pixel class prediction on a frozen foundation-model token grid:

```
SegmentationManifest -> FeatureExtractor -> train (decoder + head) -> evaluate
```

The encoder emits a **token grid per tile** (not one vector), and a **decoder**
upsamples that grid to a per-pixel map. [Detection](walkthrough-detection.ipynb)
is the same dense flow with point supervision and a different head.

> Tiny synthetic data, CPU-only, ungated encoder — the numbers are
> meaningless; the point is the API. We use
> [phikon](https://huggingface.co/owkin/phikon) at its native **224 px**
> window (a 14×14 token grid), which avoids position-embedding interpolation.

## ⚠️ Scaffolding (not soma API)

Dense supervision lives in per-sample files, not a scalar `label`: `dataset.csv`
carries `sample_id, image_path, label_mask_path`, where the mask is an integer-class
raster the same size as the ROI. We fabricate small **224 px ROI tiles** (the
dense flow consumes fixed-size tiles/ROIs, not whole WSIs) plus their masks.

In [ ]:
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from PIL import Image

WORK = Path(tempfile.mkdtemp(prefix='soma-segmentation-'))
ROIS = WORK / 'rois'; MASKS = WORK / 'masks'
for d in (ROIS, MASKS): d.mkdir()
rng = np.random.default_rng(0)

SIZE = 224          # phikon native window
SPACING = 0.5       # microns/pixel
NUM_CLASSES = 3     # 0 = background, 1, 2 = tissue classes

def make_roi(path):
    img = np.clip(np.stack([np.full((SIZE, SIZE), 150),
                            np.full((SIZE, SIZE), 70),
                            np.full((SIZE, SIZE), 160)], -1).astype(np.int16)
                  + rng.integers(-30, 30, (SIZE, SIZE, 3)), 0, 255).astype(np.uint8)
    tifffile.imwrite(path, img, photometric='rgb', tile=(SIZE, SIZE),
                     resolution=(20000, 20000), resolutionunit='CENTIMETER')

def make_mask(path):
    m = np.zeros((SIZE, SIZE), np.uint8)
    m[SIZE // 4:SIZE // 2, SIZE // 4:SIZE // 2] = 1
    m[SIZE // 2:3 * SIZE // 4, SIZE // 2:3 * SIZE // 4] = 2
    Image.fromarray(m).save(path)

ids = [f'roi{i:02d}' for i in range(8)]
for sid in ids:
    make_roi(ROIS / f'{sid}.tif')
    make_mask(MASKS / f'{sid}.png')

split = ['train'] * 4 + ['tune'] * 2 + ['test'] * 2
splits_csv = WORK / 'splits.csv'
pd.DataFrame({'sample_id': ids, 'split': split, 'fold': 0}).to_csv(splits_csv, index=False)

img_paths = [str(ROIS / f'{s}.tif') for s in ids]

seg_csv = WORK / 'seg.csv'
pd.DataFrame({'sample_id': ids, 'image_path': img_paths,
              'label_mask_path': [str(MASKS / f'{s}.png') for s in ids]}).to_csv(seg_csv, index=False)
print(pd.read_csv(seg_csv).head(3).to_string(index=False))

## 1. Extract dense token grids

`FeatureExtractor` reads each ROI at the requested spacing and runs the frozen
encoder to produce a `(feature_dim, gh, gw)` grid per sample, stored in a
`DenseFeatureStore`. With phikon at 224 px and patch-16 that's a 14×14 grid.

In [ ]:
from soma import (
    SegmentationManifest, FeatureExtractor, EncoderConfig, CacheConfig,
    PreprocessingConfig,
)

seg_manifest = SegmentationManifest(seg_csv)
extractor = FeatureExtractor(
    seg_manifest,
    EncoderConfig(name='phikon'),
    preprocessing=PreprocessingConfig(
        requested_tile_size_px=SIZE, requested_spacing_um=SPACING,
        backend='openslide',
    ),
    cache=CacheConfig(enabled=False),
    output_root=str(WORK / 'dense'),
)
dense_store = extractor.extract().source
print('dense grids for', len(dense_store.available_samples), 'ROIs')

## 2. Train the decoder + head

`train(dataset_type='segmentation', ...)` builds a **decoder**
(`lightweight_conv` upsamples the token grid) plus a parameter-free head that
crops to the mask and scores Dice / IoU. `SegmentationManifest` is the dense
counterpart of `Dataset`.

In [ ]:
from soma import (
    Splits, DecoderConfig, TaskConfig, TrainingConfig, EvalConfig,
    PreprocessingConfig, train,
)

seg_splits = Splits(splits_csv, seg_manifest)

seg_result = train(
    feature_store=dense_store,
    dataset=seg_manifest,
    splits=seg_splits,
    dataset_type='segmentation',
    decoder=DecoderConfig(name='lightweight_conv'),
    task=TaskConfig(name='segmentation', params={'num_classes': NUM_CLASSES}),
    training=TrainingConfig(epochs=3, batch_size=2, learning_rate=1e-3, seed=0),
    evaluation=EvalConfig(metrics=['mean_dice', 'mean_iou']),
    # our PNG masks carry no spacing metadata; declare the ROI spacing so they
    # register against the grids extracted at the same spacing.
    preprocessing=PreprocessingConfig(requested_spacing_um=SPACING),
    run_dir=str(WORK / 'runs' / 'segmentation'),
)
print('segmentation run dir:', seg_result.run_dir)

## 3. The one-shot `Pipeline` equivalent

`Pipeline` collapses extract + train + evaluate into a single config-driven
call.

*(Shown for reference, not executed.)*

```python
from soma import (
    Pipeline, PipelineConfig, PreprocessingConfig, EncoderConfig,
    DecoderConfig, TaskConfig, TrainingConfig, EvalConfig, CacheConfig,
)

config = PipelineConfig(
    dataset_csv=str(seg_csv),
    splits_csv=str(splits_csv),
    output_root='output/segmentation',
    dataset_type='segmentation',
    preprocessing=PreprocessingConfig(
        backend='openslide', requested_tile_size_px=224, requested_spacing_um=0.5,
    ),
    encoder=EncoderConfig(name='phikon'),
    decoder=DecoderConfig(name='lightweight_conv'),
    task=TaskConfig(name='segmentation', params={'num_classes': 3}),
    training=TrainingConfig(epochs=3, batch_size=2, learning_rate=1e-3),
    evaluation=EvalConfig(metrics=['mean_dice', 'mean_iou']),
    cache=CacheConfig(enabled=True),
)
results = Pipeline(config).run()
```